# Sesi 5 – Visualisasi Data

| Keterangan | Detail |
|---|---|
| **Nama Lengkap** | [Farraz Dirgham H] |
| **NIM** | [240401020170] |
| **Kelas** | [IF405] |
| **Mata Kuliah** | Data Science |
| **Topik** | Visualisasi Data dengan Matplotlib & Seaborn |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
np.random.seed(42)
print('Library siap!')

## 1. Dataset: Penjualan E-Commerce

In [ ]:
n = 300
bulan    = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
kategori = ['Elektronik','Pakaian','Makanan','Buku','Olahraga']
kota     = ['Jakarta','Bandung','Surabaya','Medan','Bali']
warna    = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800']

df = pd.DataFrame({
    'bulan'       : np.random.choice(bulan, n),
    'kategori'    : np.random.choice(kategori, n, p=[0.28,0.27,0.22,0.13,0.10]),
    'kota'        : np.random.choice(kota, n),
    'unit_terjual': np.random.randint(10, 500, n),
    'pendapatan'  : np.random.normal(2_500_000, 800_000, n).round(-3),
    'rating'      : np.random.uniform(3.0, 5.0, n).round(1),
    'return_rate' : np.random.uniform(0, 15, n).round(1)
})
print(f'Shape: {df.shape}')
df.head()

## 2. Bar Chart – Perbandingan Antar Kategori

In [ ]:
total = df.groupby('kategori')['unit_terjual'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(total.index, total.values, color=warna, edgecolor='white')
for bar, v in zip(bars, total.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{v:,}', ha='center', va='bottom', fontweight='bold')
ax.set_title('Total Unit Terjual per Kategori', fontsize=14, fontweight='bold')
ax.set_xlabel('Kategori')
ax.set_ylabel('Unit Terjual')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('bar_kategori.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pie & Donut Chart – Proporsi Pendapatan

In [ ]:
pk = df.groupby('kota')['pendapatan'].sum()
palette = sns.color_palette('Set2', len(pk))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie
wedges, texts, autotexts = axes[0].pie(
    pk, labels=pk.index, autopct='%1.1f%%', colors=palette,
    startangle=140, wedgeprops={'edgecolor':'white','linewidth':2})
for t in autotexts: t.set_fontweight('bold')
axes[0].set_title('Pie Chart – Pendapatan per Kota', fontweight='bold')

# Donut
axes[1].pie(pk, labels=pk.index, autopct='%1.1f%%', colors=palette,
            startangle=140, pctdistance=0.78,
            wedgeprops={'edgecolor':'white','linewidth':2,'width':0.55})
axes[1].text(0, 0, 'Pendapatan', ha='center', va='center', fontsize=11, fontweight='bold')
axes[1].set_title('Donut Chart – Pendapatan per Kota', fontweight='bold')

plt.tight_layout()
plt.savefig('pie_donut.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Histogram & KDE – Distribusi Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['pendapatan'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['pendapatan'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean = {df["pendapatan"].mean()/1e6:.2f}Jt')
axes[0].set_title('Histogram Pendapatan', fontweight='bold')
axes[0].set_xlabel('Pendapatan (Rp)')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

cmap_kat = dict(zip(kategori, warna))
for kat, c in zip(kategori, warna):
    sns.kdeplot(df[df['kategori']==kat]['pendapatan'], ax=axes[1],
                color=c, linewidth=2, label=kat, fill=True, alpha=0.1)
axes[1].set_title('KDE Pendapatan per Kategori', fontweight='bold')
axes[1].set_xlabel('Pendapatan (Rp)')
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('histogram_kde.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scatter Plot – Hubungan Dua Variabel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for kat in kategori:
    sub = df[df['kategori']==kat]
    ax.scatter(sub['unit_terjual'], sub['pendapatan'],
               color=cmap_kat[kat], alpha=0.6, s=55, edgecolors='white', lw=0.4, label=kat)
z  = np.polyfit(df['unit_terjual'], df['pendapatan'], 1)
xr = np.linspace(df['unit_terjual'].min(), df['unit_terjual'].max(), 100)
ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=2, label='Tren')
ax.set_title('Scatter: Unit Terjual vs Pendapatan', fontsize=14, fontweight='bold')
ax.set_xlabel('Unit Terjual')
ax.set_ylabel('Pendapatan (Rp)')
ax.legend(title='Kategori', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('scatter_penjualan.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Box Plot & Violin Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='kategori', y='rating', palette='Set2', ax=axes[0],
            width=0.5, flierprops={'marker':'o','markerfacecolor':'red','markersize':4})
sns.stripplot(data=df, x='kategori', y='rating', color='black',
              alpha=0.12, size=3, ax=axes[0], jitter=True)
axes[0].set_title('Box Plot: Rating per Kategori', fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

sns.violinplot(data=df, x='kategori', y='pendapatan', palette='Set1',
               ax=axes[1], inner='quartile', linewidth=1.2)
axes[1].set_title('Violin Plot: Pendapatan per Kategori', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('box_violin.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Line Chart – Tren Bulanan

In [ ]:
urutan = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
tren = df.groupby('bulan')['pendapatan'].sum().reindex(urutan)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(urutan, tren.values/1e6, 'o-', color='#2196F3',
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.fill_between(range(len(urutan)), tren.values/1e6, alpha=0.1, color='#2196F3')
ax.set_title('Tren Pendapatan Bulanan (Juta Rp)', fontsize=13, fontweight='bold')
ax.set_xlabel('Bulan')
ax.set_ylabel('Total Pendapatan (Juta Rp)')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('line_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Heatmap – Korelasi & Pivot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df[['unit_terjual','pendapatan','rating','return_rate']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            square=True, linewidths=1, ax=axes[0], cbar_kws={'shrink':0.8})
axes[0].set_title('Heatmap Korelasi', fontweight='bold')

pivot = df.pivot_table(values='rating', index='kategori', columns='kota', aggfunc='mean')
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[1], cbar_kws={'shrink':0.8})
axes[1].set_title('Rata-rata Rating: Kategori x Kota', fontweight='bold')

plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Dashboard Multi-Panel

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
t   = df.groupby('kategori')['unit_terjual'].sum().sort_values()
ax1.barh(t.index, t.values, color=warna[::-1], edgecolor='white')
ax1.set_title('Unit Terjual per Kategori', fontweight='bold')

ax2 = fig.add_subplot(gs[0, 1])
pk2 = df.groupby('kota')['pendapatan'].sum()
ax2.pie(pk2, labels=pk2.index, autopct='%1.0f%%',
        colors=sns.color_palette('Pastel1', len(pk2)))
ax2.set_title('Pendapatan per Kota', fontweight='bold')

ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(df['rating'], bins=15, color='mediumpurple', edgecolor='white')
ax3.set_title('Distribusi Rating', fontweight='bold')

ax4 = fig.add_subplot(gs[1, 0:2])
for kat in kategori:
    sub = df[df['kategori']==kat]
    ax4.scatter(sub['unit_terjual'], sub['pendapatan'],
                color=cmap_kat[kat], alpha=0.5, s=35, label=kat)
ax4.set_title('Unit Terjual vs Pendapatan', fontweight='bold')
ax4.legend(fontsize=8)

ax5 = fig.add_subplot(gs[1, 2])
sns.boxplot(data=df, x='kategori', y='rating', palette='Set2', ax=ax5, width=0.5)
ax5.set_title('Rating per Kategori', fontweight='bold')
ax5.set_xticklabels(ax5.get_xticklabels(), rotation=15, fontsize=8)

fig.suptitle('Dashboard Penjualan E-Commerce', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard selesai!')

## 10. Kesimpulan

| Jenis Visualisasi | Kapan Digunakan |
|---|---|
| Bar Chart | Membandingkan nilai antar kategori |
| Pie / Donut | Menampilkan proporsi/komposisi |
| Histogram + KDE | Distribusi satu variabel numerik |
| Scatter Plot | Hubungan dua variabel numerik |
| Box Plot | Distribusi & outlier per kelompok |
| Violin Plot | Distribusi bentuk penuh per kelompok |
| Line Chart | Tren data berurutan/waktu |
| Heatmap | Korelasi atau agregasi dua dimensi |
| Dashboard | Ringkasan analisis menyeluruh |

> **Insight:** Pilihan visualisasi yang tepat bergantung pada **tujuan komunikasi** dan **jenis data**. Visualisasi yang baik menjawab pertanyaan bisnis sebelum pembaca sempat bertanya.